# Transfer Learning for text data using Transformer

In [ ]:
# Install dependencies
!pip install transformers datasets evaluate scikit-learn pandas > /dev/null 2>&1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


## Download Dataset

In [ ]:

import os

os.environ['KAGGLE_USERNAME'] = "abhishek072891"
os.environ['KAGGLE_KEY'] = "KGAT_ca4c8d4bd81203c895e1f5768bac947d"

!kaggle datasets download -d abhi8923shriv/sentiment-analysis-dataset
!unzip sentiment-analysis-dataset.zip -d sentiment_data

Dataset URL: https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset
License(s): CC0-1.0
100% 54.4M/54.4M [00:07<00:00, 8.15MB/s]

Archive:  sentiment-analysis-dataset.zip
  inflating: sentiment_data/test.csv  
  inflating: sentiment_data/testdata.manual.2009.06.14.csv  
  inflating: sentiment_data/train.csv  
  inflating: sentiment_data/training.1600000.processed.noemoticon.csv  


## Data Loading and Preprocessing

In [ ]:

import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import os

# Load dataset
csv_file = [f for f in os.listdir('sentiment_data') if f.endswith('.csv')][0]
df = pd.read_csv(f'sentiment_data/{csv_file}', encoding='latin-1')

# Standardize columns to 'text' and 'label'
if 'comment' in df.columns: df.rename(columns={'comment': 'text'}, inplace=True)
if 'sentiment' in df.columns: df.rename(columns={'sentiment': 'label'}, inplace=True)

# Map labels to integers if they are strings (0: Negative, 1: Neutral, 2: Positive)
if df['label'].dtype == object:
    label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
    df['label'] = df['label'].str.lower().map(label_mapping)

df = df.dropna()

df['label'] = df['label'].astype(int)

sample_size = min(10000, len(df))
print(f"Dataset has {len(df)} valid rows. Sampling {sample_size} rows for training.")
df = df.sample(sample_size, random_state=42)

# Split data
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

Dataset has 3534 valid rows. Sampling 3534 rows for training.


Map:   0%|          | 0/2827 [00:00<?, ? examples/s]

Map:   0%|          | 0/707 [00:00<?, ? examples/s]

## Model Training

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

# Start fine-tuning
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,No log,0.638391
2,No log,0.640169
3,0.576331,0.649227


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=531, training_loss=0.5666053146965759, metrics={'train_runtime': 113.0258, 'train_samples_per_second': 75.036, 'train_steps_per_second': 4.698, 'total_flos': 280869010811136.0, 'train_loss': 0.5666053146965759, 'epoch': 3.0})

## Evaluation & Classification Report

In [ ]:

# Get predictions
predictions = trainer.predict(tokenized_val)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("\nClassification Report (Text Transfer Learning):")
print(classification_report(labels, preds, target_names=["Negative", "Neutral", "Positive"]))


Classification Report (Text Transfer Learning):
              precision    recall  f1-score   support

    Negative       0.73      0.79      0.76       207
     Neutral       0.70      0.69      0.70       278
    Positive       0.82      0.77      0.79       222

    accuracy                           0.74       707
   macro avg       0.75      0.75      0.75       707
weighted avg       0.75      0.74      0.74       707

